# 이게 미술 재료라고? - MoMA CSV 챗봇

MoMA 작품 데이터에서 제목, 작가, 재료, 작품 분류를 검색하는 챗봇입니다.

- 원본 데이터: [MoMA Collection](https://github.com/MuseumofModernArt/collection)
- 사용 데이터: 원본 중 5,000개 작품을 뽑은 CSV
- 검색 예시: `rubber`, `chocolate`, `neon`, `hair`, `machine`


## 1. 데이터 불러오기

저장소의 CSV 파일을 먼저 찾고, Colab에서는 GitHub에 있는 파일을 불러옵니다.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/JuneKunst/physical-ai-project/main/mini-kaggle-chatbot/data/moma_artworks_sample.csv"
local_paths = [
    Path("data/moma_artworks_sample.csv"),
    Path("mini-kaggle-chatbot/data/moma_artworks_sample.csv"),
]

data_path = next((path for path in local_paths if path.exists()), None)
df = pd.read_csv(data_path if data_path else DATA_URL)

print("데이터 크기:", df.shape)
print("\n컬럼 목록:")
print(df.columns.tolist())
df.head()


## 2. 검색에 사용할 컬럼 정하기

작품 제목을 중심으로 작가, 재료, 분류, 부서, 국적을 함께 검색합니다.
빈 값은 `정보없음`으로 채웠습니다.


In [ ]:
title_column = "title"
search_columns = [
    "title",
    "artist",
    "medium",
    "classification",
    "department",
    "nationality",
]
answer_columns = ["artist", "date", "medium", "classification", "department"]

print(df[search_columns].isnull().sum())

for column in search_columns + ["url", "image_url"]:
    df[column] = df[column].fillna("정보없음").astype(str)

print("\n결측치 처리 후:")
print(df[search_columns].isnull().sum())


## 3. 작품 검색 함수 만들기

영문 키워드를 기본으로 검색합니다. 자주 사용할 만한 단어는 한국어로 입력해도 찾을 수 있게 바꿔줍니다.


In [ ]:
keyword_aliases = {
    "고무": "rubber",
    "머리카락": "hair",
    "초콜릿": "chocolate",
    "음식": "food",
    "네온": "neon",
    "기계": "machine",
    "플라스틱": "plastic",
    "유리": "glass",
    "천": "fabric",
    "소리": "sound",
    "장난감": "toy",
    "비누": "soap",
    "먼지": "dust",
    "뼈": "bone",
    "피": "blood",
}


def search_data(keyword, top_n=3):
    keyword = str(keyword).strip().lower()
    if not keyword:
        return None

    keyword = keyword_aliases.get(keyword, keyword)
    mask = df[search_columns].apply(
        lambda column: column.str.lower().str.contains(keyword, na=False, regex=False)
    ).any(axis=1)

    results = df[mask].copy()
    if results.empty:
        return None

    results["has_image"] = results["image_url"].str.startswith("https://")
    results = results.sort_values("has_image", ascending=False, kind="stable")
    return results.drop(columns="has_image").head(top_n)


In [ ]:
for keyword in ["rubber", "초콜릿", "neon", "없는재료"]:
    result = search_data(keyword)
    count = 0 if result is None else len(result)
    print(f"{keyword}: {count}개")


## 4. 챗봇 답변 만들기

검색된 작품의 이미지와 제목, 작가, 제작 시기, 재료, 작품 분류를 보여줍니다.


In [ ]:
def shorten(text, limit=220):
    text = str(text)
    return text if len(text) <= limit else text[:limit].rstrip() + "..."


def format_answer(results):
    if results is None:
        return "관련된 작품을 찾지 못했습니다. 다른 재료나 작품 이름으로 검색해 주세요."

    lines = [f"찾은 작품 중 {len(results)}개를 보여드릴게요.\n"]
    for _, row in results.iterrows():
        lines.append(f"### {row['title']}")
        if row["image_url"].startswith("https://"):
            lines.append(f"![{row['title']}]({row['image_url']})")
        lines.append(f"- 작가: {row['artist']}")
        lines.append(f"- 제작 시기: {row['date']}")
        lines.append(f"- 재료: {shorten(row['medium'])}")
        lines.append(f"- 분류: {row['classification']}")
        lines.append(f"- 담당 부서: {row['department']}")
        if row["url"] != "정보없음":
            lines.append(f"- [MoMA 작품 페이지]({row['url']})")
        lines.append("")
    return "\n".join(lines)


def chatbot(user_input):
    return format_answer(search_data(user_input))


print(chatbot("고무"))


## 5. 반복해서 질문하기

터미널 방식으로 확인하려면 아래 함수를 실행한 뒤 `run_terminal_chatbot()`을 호출합니다.


In [ ]:
def run_terminal_chatbot():
    print("작품 이름이나 재료를 입력해 주세요. 종료하려면 '종료'를 입력하세요.")

    while True:
        user_input = input("\n나: ").strip()
        if user_input == "종료":
            print("챗봇을 종료합니다.")
            break
        print("\n챗봇:")
        print(chatbot(user_input))


## 6. Gradio 채팅 화면 만들기

마지막 셀을 실행하면 Colab 안에 채팅 화면이 나타나고, 외부에서 접속할 수 있는 임시 링크도 만들어집니다.


In [ ]:
!pip -q install gradio


In [ ]:
import gradio as gr


def gradio_chatbot(message, history):
    return chatbot(message)


demo = gr.ChatInterface(
    fn=gradio_chatbot,
    title="이게 미술 재료라고?",
    description="MoMA 작품의 이미지, 제목, 작가와 재료를 검색해 보세요.",
    examples=["rubber", "초콜릿", "neon", "머리카락", "machine"],
)


In [ ]:
demo.launch(share=True)


## 테스트한 질문

- `rubber`: 고무가 사용된 작품
- `초콜릿`: 초콜릿이 언급된 작품
- `neon`: 네온을 사용한 작품
- `머리카락`: 머리카락이 언급된 작품
- `machine`: 기계와 관련된 작품
- `없는재료`: 검색 결과가 없을 때 안내 문구 확인
